In [1]:
import networkx as nx
import numpy as np
import pnm
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
import csv
print(nx.__version__)

3.2.1


In [2]:
folder = os.path.join(os.getcwd())
folder = os.getcwd()
netfile = "20241003-115534_GM1-NP500-2.18x2.18x2.18-p36.82-SR0.40-PT_RATIO2.00.pnm"
#filenames = [os.path.join(folder,f) for f in os.listdir(folder) if os.path.isfile(os.path.join(folder,f)) if f.endswith(".pnm")]

print('Loading the network...',end="")
pn  = pnm.pore_network.read(os.path.join(folder,"networks", netfile), method="pickle")
print("OK")
print("{} pores and {}  throats".format(pn.graph.number_of_nodes(),pn.graph.number_of_edges()))

Loading the network...OK
500 pores and 1761  throats


Note : d'après T. Sanchez le coeff effectif relatif de diffusion moléculaire pour des pâtes de type CEM-V est de l'ordre de 2.6e-4 à 2.8e-4. Ca parait assez faible / coeff diff sur des mortiers ?

In [3]:
print('Setting up the simulation...',end="")
scale = 1e-6
#pnm.scale(pn,scale)
BC1 = 1
BC2 = 0
Dinf = 1.25e-5
Ma = 44e-3 #Masse atomique en kg/mol
DKNUDSEN = (1/6)*np.sqrt((8*8.3144*293)/(np.pi*Ma))
S = {'x':np.prod(pn.graph.graph['extent'][1:]),
    'y':np.prod(pn.graph.graph['extent'][0::2]),
    'z':np.prod(pn.graph.graph['extent'][:2])}
L = {'x':pn.graph.graph['extent'][0],
    'y':pn.graph.graph['extent'][1],
    'z':pn.graph.graph['extent'][2]}
Deff = {}

throat_model = "throat"
diff_model = "All" #"Fick"
print("OK")

nbcpores = {}
for direction in ['x','y','z']:
    dir1 = direction + "+"
    dir2 = direction + "-"
    nbcpores[direction] = 100*(len(pn.get_pores_by_category([dir1])) +
            len(pn.get_pores_by_category([dir2])) ) / pn.graph.number_of_nodes()
    print(nbcpores[direction])

Setting up the simulation...OK
0.4
0.4
0.4


In [4]:
for direction in ['z']:

    dir1 = direction + "+"
    dir2 = direction + "-"

#Note that when throat radius is "effective" the sum of pore throats surface can be greater than the network surface...
#which can lead to Deff>Dinf
    solver = pnm.molecular_diffusion(pn)
    solver.setup_throat_model(model=throat_model)
    solver.setup_diffusivity_model(model=diff_model,d_fick=Dinf,d_knudsen = DKNUDSEN, scale = 1/scale)
    solver.add_bc(BC1,pn.get_pores_by_category([dir1],mode='match_one'),bc_type='Dirichlet')
    solver.add_bc(BC2,pn.get_pores_by_category([dir2],mode='match_one'),bc_type='Dirichlet')
    print('Building coeff matrix...',end="")
    solver.setup_coef_matrix()
    print("OK")
    print("Solving...",end="")
    solver.solve(solver='cg',tol=1e-8)
    print('OK (status {})'.format(solver.iter))
    q1 = solver.compute_boundary_flow(pn.get_pores_by_category([dir1],mode='match_one'))
    q2 = solver.compute_boundary_flow(pn.get_pores_by_category([dir2],mode='match_one'))
    print("Flow in", -q1," Flow out:",q2)
    Deff[direction] = q1*L[direction]*scale**2 /((BC2-BC1)*S[direction])
    print('Deff {} = {:.3e}, Deff/Dinf = {:.3e}'.format(direction,Deff[direction] ,Deff[direction]/Dinf))

    #nbcporeslabels = pn.get_pores_by_category([dir2]).extend(pn.get_pores_by_category([dir1]))
    #nbcpores[direction] = sum([pn.graph.nodes[n]['volume'] for n in nbcporeslabels]) / pn.pores_volume
    #nbcpores[direction] = 100*(len(pn.get_pores_by_category([dir1])) +
    #        len(pn.get_pores_by_category([dir2])) ) / pn.graph.number_of_nodes()


Using  throat  radius model
Using  All  diffusivity model(s)
Building coeff matrix...OK
Solving...OK (status 0)
Flow in 63.70224013317866  Flow out: 63.70224013637456
Deff z = 2.919e-11, Deff/Dinf = 2.335e-06


In [6]:
name = throat_model + "-" + diff_model + "-" + os.path.split(netfile)[-1][:-4]
avgc = sum([pn.graph.degree(n) for n in pn.graph.nodes])/pn.graph.number_of_nodes()

with open(name+".csv",'w') as csvfile:
    csvwriter = csv.writer(csvfile, delimiter=';', dialect='excel')
    csvwriter.writerow(['name','nnodes','nthroats','porosity','avg connectivity','throat model',
                        '%BCP-x','%BCP-y','%BCP-z','Deff-x','Deff-y','Deff-z'])
    csvwriter.writerow([name,
                        pn.graph.number_of_nodes(),
                        pn.graph.number_of_edges(),
                        pn.graph.graph['porosity'],
                        avgc,throat_model,
                        nbcpores['x'],nbcpores['y'],nbcpores['z'],
                        Deff['x'],Deff['y'],Deff['z']])

In [8]:
print (avgc)

5.90944


In [ ]:
c = solver.get_pores_concentration_dict()
norm = mpl.colors.Normalize(vmin=1, vmax=2)
pmap = {k:np.array(mpl.cm.jet(norm(c[k]))[:-1]) for k in c}
cmap =[0.5*(pmap[n2]+pmap[n1]) for n1,n2 in pn.graph.edges]

In [1]:
#radii = nx.get_node_attributes(pn.graph,'radius').values()
#rmax = max(radii)
#rmin = min(radii)

#norm = mpl.colors.Normalize(vmin=rmin, vmax=rmax)
#cmap = mpl.cm.jet
#pores_cm = {n:cmap(norm(r))[:-1] for (n,r) in nx.get_node_attributes(pn.graph,'radius').items()}
#throats_cm = [cmap(norm(r))[:-1]]
#for i in pn.get_pores_by_category(['x+','x-']):
#    pores_cm[i] = [1,0,0]
pmap = {n:[1,0,0] for n in pn.get_pores_by_category(['y+','y-'])}
viewer = pnm.PNM_3d_viewer(pn)
viewer.make_geometry()
viewer.maps_colors(pores_colormap = pmap, throats_colormap=cmap)
viewer.render()

NameError: name 'pn' is not defined

In [44]:
viewer.save('doublescale.jpg')

In [4]:
pmap = {n:[1,0,0] for n in pn.get_pores_by_category(['x+','x-'])}
viewer = pnm.PNM_3d_viewer(pn)
viewer.make_geometry(render_throats=False)
viewer.maps_colors(pores_colormap = pmap,render_throats=False)
viewer.render()